# 01 — Bronze Ingest: SaaS Product & Customer Analytics
Reads the four raw CSV files from the injected data-source path and writes them as Delta tables in the lakehouse.

| Table | Source |
|---|---|
| bronze_accounts | accounts.csv |
| bronze_users | users.csv |
| bronze_events | events.csv |
| bronze_support_tickets | support_tickets.csv |


In [ ]:
# Read the raw CSVs from the path injected by the deployer (a Shortcut or the
# uploaded landing folder). Tables are written to the attached default lakehouse
# (dbo schema), matching every other sector's medallion notebooks.
RAW_PATH = "{{DATA_SOURCE_PATH}}"


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)


In [ ]:
# ── accounts ─────────────────────────────────────────────────────────────────
accounts_schema = StructType([
    StructField("account_id",      StringType(),  False),
    StructField("plan",            StringType(),  True),
    StructField("mrr_usd",         IntegerType(), True),
    StructField("industry",        StringType(),  True),
    StructField("region",          StringType(),  True),
    StructField("signup_date",     StringType(),  True),
    StructField("churn_date",      StringType(),  True),
    StructField("is_churned",      IntegerType(), True),
    StructField("seat_count",      IntegerType(), True),
    StructField("health_score",    DoubleType(),  True),
])

df_acc = (
    spark.read.option("header", True).schema(accounts_schema)
    .csv(f"{RAW_PATH}/accounts.csv")
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("churn_date",  F.to_date("churn_date"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_acc.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("bronze_accounts")
print(f"bronze_accounts: {df_acc.count():,}")

In [ ]:
# ── users ────────────────────────────────────────────────────────────────────
users_schema = StructType([
    StructField("user_id",              StringType(),  False),
    StructField("account_id",           StringType(),  True),
    StructField("role",                 StringType(),  True),
    StructField("is_active",            IntegerType(), True),
    StructField("last_login_date",      StringType(),  True),
    StructField("signup_date",          StringType(),  True),
    StructField("logins_last_30_days",  IntegerType(), True),
])

df_usr = (
    spark.read.option("header", True).schema(users_schema)
    .csv(f"{RAW_PATH}/users.csv")
    .withColumn("last_login_date", F.to_date("last_login_date"))
    .withColumn("signup_date",     F.to_date("signup_date"))
    .withColumn("_ingested_at",    F.current_timestamp())
)

df_usr.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("bronze_users")
print(f"bronze_users: {df_usr.count():,}")

In [ ]:
# ── events ───────────────────────────────────────────────────────────────────
events_schema = StructType([
    StructField("event_id",      StringType(),  False),
    StructField("user_id",       StringType(),  True),
    StructField("account_id",    StringType(),  True),
    StructField("event_date",    StringType(),  True),
    StructField("feature",       StringType(),  True),
    StructField("action",        StringType(),  True),
    StructField("session_id",    StringType(),  True),
    StructField("duration_secs", IntegerType(), True),
])

df_ev = (
    spark.read.option("header", True).schema(events_schema)
    .csv(f"{RAW_PATH}/events.csv")
    .withColumn("event_date",  F.to_date("event_date"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_ev.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("bronze_events")
print(f"bronze_events: {df_ev.count():,}")

In [ ]:
# ── support_tickets ──────────────────────────────────────────────────────────
tickets_schema = StructType([
    StructField("ticket_id",       StringType(),  False),
    StructField("account_id",      StringType(),  True),
    StructField("created_at",      StringType(),  True),
    StructField("resolved_at",     StringType(),  True),
    StructField("category",        StringType(),  True),
    StructField("priority",        StringType(),  True),
    StructField("resolution_hrs",  DoubleType(),  True),
    StructField("sla_target_hrs",  IntegerType(), True),
    StructField("is_sla_breached", IntegerType(), True),
    StructField("csat_score",      IntegerType(), True),
])

df_tk = (
    spark.read.option("header", True).schema(tickets_schema)
    .csv(f"{RAW_PATH}/support_tickets.csv")
    .withColumn("created_at",  F.to_timestamp("created_at"))
    .withColumn("resolved_at", F.to_timestamp("resolved_at"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_tk.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("bronze_support_tickets")
print(f"bronze_support_tickets: {df_tk.count():,}")

In [ ]:
print("\n=== Bronze Ingest Complete ===")
for tbl in ["bronze_accounts", "bronze_users", "bronze_events", "bronze_support_tickets"]:
    cnt = spark.table(tbl).count()
    print(f"  {tbl}: {cnt:,}")